## LangGraph-Orchestrated LLM-Based Optimization Agent

This implements the **LLM-based autonomous optimization agent** following the **Always Calling a Tool First** design pattern.

**LangGraph** orchestrates the optimization workflow by always executing the deterministic **clustering** and **evaluation** tools before invoking the LLM. Rather than performing clustering itself, the LLM acts exclusively as the **reasoning component**, analysing the optimization history and evaluation metrics to recommend the next set of clustering parameters.

The objective of this phase is not to find the optimal clustering parameters, but to validate that an LLM can function as the reasoning layer within an iterative optimization framework while deterministic tools perform the computational analysis.

Specifically, this phase verifies that the agent can:

* Automatically execute the clustering and evaluation tools through LangGraph.
* Always call deterministic tools before invoking the LLM.
* Observe and evaluate clustering quality using the internal evaluation metrics.
* Remember previous optimization experiments through the graph state.
* Reason over the optimization history using GPT-5.
* Return validated parameter suggestions using a **Pydantic** structured output model.
* Decide whether the optimization should continue or stop.

### Implementing Guardrails

To ensure the optimization remains efficient, reproducible, and computationally bounded, the workflow introduces its first deterministic guardrail:

* **Maximum iteration limit**

Rather than allowing the LLM to continue proposing new experiments indefinitely, the framework enforces a predefined optimization budget. The guardrail is evaluated **before** the LLM is invoked, preventing unnecessary API calls and reducing execution time from approximately **686 seconds** to **90 seconds** during testing.

### Current Architecture

At this stage, the optimization framework combines deterministic execution with adaptive LLM reasoning:

* **LangGraph** orchestrates the optimization workflow.
* **Deterministic clustering and evaluation tools** are always executed first.
* **Deterministic guardrails** govern the optimization process and constrain the search.
* **GPT-5** serves as the reasoning component, interpreting the optimization history and evaluation metrics to propose the next clustering parameters.
* **Pydantic** validates the structured output returned by the LLM before it is incorporated into the graph state.

Future phases will extend this architecture with additional guardrails, duplicate parameter detection, experiment logging, stability analysis, marker gene analysis, and automated optimization reporting.


## Modules Used in This Phase

| Module                                   | Purpose                                                                                                                                                                                     |
| ---------------------------------------- | ------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| **state.py**                             | Defines the LangGraph state, including the dataset (`AnnData`), clustering parameters, evaluation metrics, optimization history, iteration counter, decision, and reasoning.                |
| **clustering.py**                        | Implements the deterministic clustering tool. Runs the neighborhood graph construction and Leiden clustering using the current optimization parameters.                                     |
| **evaluation.py**                        | Computes the clustering evaluation metrics (e.g., Silhouette Score and Davies–Bouldin Index) and records each optimization experiment in the graph state.                                   |
| **agent.py**                             | Defines the graph nodes (`cluster_node`, `evaluate_node`, and `reflection_node`). The reflection node performs LLM-based reasoning, applies guardrails, and updates the optimization state. |
| **graph.py**                             | Builds the LangGraph workflow, connects the nodes, defines the conditional optimization loop, and compiles the executable graph.                                                            |
| **decision.py**                          | Defines the Pydantic models (`SuggestedParameters` and `OptimizationDecision`) used to validate the structured output returned by the LLM.                                                  |
| **prompts.py**                           | Contains the prompt template used by the reflection node. It formats the optimization history, parameters, and evaluation metrics into a prompt for the LLM.                                |
| **openAI_llm.py** *(or `llm.py`)*        | Configures the GPT-5 model used for structured reasoning during the reflection step.                                                                                                        |
| **optimization.ipynb** *(this notebook)* | Initializes the graph state, executes the LangGraph workflow, validates the optimization results, and demonstrates the complete LLM-driven optimization loop.                               |


In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from dotenv import load_dotenv
import os

load_dotenv(override=True)

print(os.getenv("OPENAI_API_KEY") is not None)

True


In [3]:
from src import data_load as dl
from src import preprocessing_pipeline as prp
from src import clustering as cl
from src import evaluation as ev
from src import optimization as opt
from src import visualisation as viz
from src import marker as mk
from src.graph import graph
from src.agent import cluster_node, evaluate_node, manual_reflection_node, reflection_node
from src.decision import OptimizationDecision
from src.prompts import build_reflection_prompt

import scanpy as sc
import pandas as pd
from langchain_openai.chat_models import ChatOpenAI 

### Create the LLM (OPENAI)

In [4]:
llm = ChatOpenAI(model="gpt-5", temperature=0,)

In [5]:
structured_llm = llm.with_structured_output(OptimizationDecision) #This help to return an optimized decision

## Fixed Preprocessing Pipeline

In [6]:
pbmc_data = dl.load_pbmc3k()
pbmc_data = prp.run_preprocessing_pipeline(pbmc_data)

Running QC...
Running normalization...
Running PCA...
Finished preprocessing.


In [7]:
pbmc_data.var["highly_variable"].sum()

np.int64(2000)

## Agent State Setup

In [8]:
state = {
    "messages": [],
    "adata": pbmc_data,

    "parameters": {
        "n_neighbors": 15,
        "resolution": 1.0,
        "n_pcs": 20,
    },

    "metrics": {},
    "history": [],
    "iteration": 0,
    "decision": "continue",
    "reason": "",
}

In [9]:
result = graph.invoke(state)

In [10]:
history = result["history"]

parameters = result["parameters"]

metrics = result["metrics"]

In [11]:
state.keys()

dict_keys(['messages', 'adata', 'parameters', 'metrics', 'history', 'iteration', 'decision', 'reason'])

In [12]:
prompt = build_reflection_prompt(
    history=history,
    parameters=parameters,
    metrics=metrics,) 


In [13]:
decision = structured_llm.invoke(prompt)

In [14]:
print(type(decision))

<class 'src.decision.OptimizationDecision'>


In [15]:
# print(decision)

In [16]:
print(decision.parameters.n_neighbors)
print(decision.parameters.resolution)
print(decision.parameters.n_pcs)

print(decision.reason)

20
0.4
30
Resolution ~0.4 performed best so far; changing n_neighbors from 15→20 had only minor impact. Unexplored n_pcs is likely a higher-leverage knob—raising to 30 can capture more biological variance and improve separation, potentially increasing silhouette and lowering DBI while altering only one parameter.


In [17]:
#print(result["history"][0]["parameters"])

In [18]:
for experiment in result["history"]:
    print(
        experiment["iteration"],
        experiment["parameters"],
        experiment["metrics"]["silhouette_score"],
        experiment["metrics"]["davies_bouldin_score"],
    )

0 {'n_neighbors': 15, 'resolution': 1.0, 'n_pcs': 20} 0.07478884607553482 2.666380841974742
1 {'n_neighbors': 15, 'resolution': 0.6, 'n_pcs': 20} 0.14038828015327454 2.160339519035311
2 {'n_neighbors': 15, 'resolution': 0.4, 'n_pcs': 20} 0.16863155364990234 1.9324675655848067
3 {'n_neighbors': 15, 'resolution': 0.3, 'n_pcs': 20} 0.16749389469623566 2.00298246460236
4 {'n_neighbors': 15, 'resolution': 0.4, 'n_pcs': 20} 0.16863155364990234 1.9324675655848067
5 {'n_neighbors': 20, 'resolution': 0.4, 'n_pcs': 20} 0.16731047630310059 1.9387778680107135


This shows the LLM is:

+ reading optimization history,
+ identifying the best experiment,
+ avoiding changing multiple parameters at once

**But the LLM occasionally proposed parameter combinations that had already been evaluated**

### Current State Info

In [19]:
print(result["iteration"])
print(result["decision"])

5
stop


The iteration being 5 shows that the agent stopped at the sixth iteraction as designed.